# Module 4: Parallel Fork-Join

Apply **Pattern 2.** Fork independent sub-tasks to run simultaneously, then merge results: latency drops to the slowest single worker.

![Parallel Fork-Join: Researcher → GraphBuilder parallel topology (Analyzer A, B, C simultaneously) → Synthesizer → Leadership Memo](./architecture.png)

**Strands pattern:** `GraphBuilder` with parallel topology: nodes with the same predecessor and no dependencies between them run in parallel automatically. No `asyncio` code needed.

**When to use this pattern:**
- Sub-tasks are independent
- Latency matters
- You want the speedup of parallelism without threading code

**Prerequisites:** Modules 1–3. This module reuses tools from Module 2.

## Tools and Components in This Module

| Component | Type | What it does |
|-----------|------|-------------|
| `get_company_data` | Tool | NovaCart financial/operational data |
| `get_market_benchmarks` | Tool | E-commerce industry benchmarks |
| `get_competitor_data` | Tool | Competitor premium tier details |
| `researcher` | Graph node (entry) | Gathers shared context with tools: no incoming edges, receives the task directly |
| `analyzer_a/b/c` | Graph nodes (parallel) | Each evaluates one option: no dependency between them, run in parallel |
| `synthesizer` | Graph node (join) | Receives all three analyses: waits for A, B, C then writes the memo |

> **How context propagates:** `GraphBuilder` automatically builds each node's input from (1) the original task and (2) the outputs of all its predecessors. No manual `str(result)` passing needed.

In [ ]:
%pip install -r requirements.txt

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1, Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2, Claude Haiku 4.5 (faster):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3, Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")

In [ ]:
import sys, os, time

# decision_brief_tools.py lives in module 02 (it defines the mock data tools used across modules).
# We add that directory to the path here so this notebook can import it directly
# without duplicating the file. In production each specialist has its own copy (see specialists/researcher/).
sys.path.insert(0, os.path.join(os.getcwd(), "..", "02-single-agent"))

from strands import Agent
from strands.multiagent import GraphBuilder
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data

---

## Part 1: System Prompts

Each node has a focused prompt. The Researcher is the entry point: it receives the original task directly. The Analyzers receive: original task + Researcher output (automatic via GraphBuilder). The Synthesizer receives: original task + all four outputs.

In [ ]:
RESEARCHER_PROMPT = (
    "You are a market research specialist. Use your tools to gather relevant data. "
    "Return structured findings. data only, no recommendations."
)

# Each Analyzer has a different option assigned in its prompt
# GraphBuilder passes the Researcher findings automatically in the context
ANALYZER_A_PROMPT = (
    "You are a business analyst. Evaluate Option A ($19.99/mo, invite-only top 10% spenders) "
    "based on the decision brief and research context provided. "
    "Return: strengths, weaknesses, complexity (Low/Med/High), top 2 risks+mitigations, verdict. "
    "150 words max."
)
ANALYZER_B_PROMPT = (
    "You are a business analyst. Evaluate Option B ($14.99/mo, 5% A/B pilot with kill-switch) "
    "based on the decision brief and research context provided. "
    "Return: strengths, weaknesses, complexity (Low/Med/High), top 2 risks+mitigations, verdict. "
    "150 words max."
)
ANALYZER_C_PROMPT = (
    "You are a business analyst. Evaluate Option C ($12.99/mo, full launch, 30-day trial) "
    "based on the decision brief and research context provided. "
    "Return: strengths, weaknesses, complexity (Low/Med/High), top 2 risks+mitigations, verdict. "
    "150 words max."
)

SYNTHESIZER_PROMPT = (
    "You are an executive communications specialist. Write a leadership memo from all analyses:\n"
    "## Recommendation (one sentence: which option and why)\n"
    "## Options at a Glance (table comparing A, B, C)\n"
    "## Top 3 Risks with mitigations\n"
    "## Success Metrics (at least 2 KPIs with numeric targets)\n"
    "## Decision Required (owner, deadline, approvers)\n"
    "Under 400 words."
)

---

## Part 2: Build the Graph

`GraphBuilder` nodes with the same predecessor and no dependency between them are executed in parallel automatically. No `asyncio` needed.

In [ ]:
DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Options:
  Option A: Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B: Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C: Full Launch: open to all users immediately, $12.99/mo + 30-day trial

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

# Create agents, one per graph node
researcher  = Agent(name="researcher",  system_prompt=RESEARCHER_PROMPT,
                    tools=[get_company_data, get_market_benchmarks, get_competitor_data],
                    callback_handler=None)
analyzer_a  = Agent(name="analyzer_a",  system_prompt=ANALYZER_A_PROMPT, callback_handler=None)
analyzer_b  = Agent(name="analyzer_b",  system_prompt=ANALYZER_B_PROMPT, callback_handler=None)
analyzer_c  = Agent(name="analyzer_c",  system_prompt=ANALYZER_C_PROMPT, callback_handler=None)
synthesizer = Agent(name="synthesizer", system_prompt=SYNTHESIZER_PROMPT, callback_handler=None)

# Build the parallel graph
builder = GraphBuilder()
builder.add_node(researcher,  "researcher")
builder.add_node(analyzer_a,  "analyzer_a")
builder.add_node(analyzer_b,  "analyzer_b")
builder.add_node(analyzer_c,  "analyzer_c")
builder.add_node(synthesizer, "synthesizer")

# Fork: researcher → A, B, C  (no edges between A/B/C → GraphBuilder runs them in parallel)
builder.add_edge("researcher", "analyzer_a")
builder.add_edge("researcher", "analyzer_b")
builder.add_edge("researcher", "analyzer_c")

# Join: A, B, C → synthesizer  (waits for all three before executing)
builder.add_edge("analyzer_a", "synthesizer")
builder.add_edge("analyzer_b", "synthesizer")
builder.add_edge("analyzer_c", "synthesizer")

builder.set_execution_timeout(300)
graph = builder.build()

---

## Part 3: Run the Graph

In [ ]:
t0 = time.time()
result = graph(DECISION_BRIEF)
elapsed = time.time() - t0

# GraphBuilder guarantees: analyzer_a, analyzer_b, analyzer_c all finish before synthesizer
# The order AMONG A/B/C may vary, no dependency between them

---

## Part 4: Inspect the Final Memo

In [ ]:
# Access individual node results
for node in reversed(result.execution_order):
    if node.node_id == "synthesizer":
        break

In [ ]:
# Token usage per node
for node in result.execution_order:
    # GraphBuilder NodeResult, metrics available via the underlying agent

# Total from accumulated metrics
usage = result.accumulated_usage if hasattr(result, 'accumulated_usage') else {}